# Imports

In [ ]:
import torch
import pandas as pd
from torch.utils.data import Dataset
from PIL import Image
from torchvision.transforms import v2
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import json
from pathlib import Path
import torch.nn as nn
import gc
from collections import Counter

from src.dataset import CreatureDataset

# Configuration

In [ ]:
MODEL_PATH = "trained_models/EfficientNet_b2/e_40_tl_235.969_vl_240.568.pth"
DATASET = "all_cards"
batch_size = 64
OUTPUT_PATH = "final_models/EfficientNet_b2"
device = device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model

In [ ]:
checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=False)
model = checkpoint["model"]
all_types = checkpoint["all_types"]
model = model.to(device)
model.eval()

print(len(all_types), all_types, sep=" types: ")

# Build dataloader
- load CSV
- define dataset+dataloader

In [ ]:
# load CSV
df = pd.read_csv(f"data/{DATASET}_manifest.csv")

dataset = CreatureDataset(df, all_types=all_types, is_training=False)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

print(len(dataset), "data entries")

# Collect all model predictions

In [ ]:
all_preds = []
all_labels = []

# just loop over full dataset and collect all predictions
with torch.no_grad():
    for images, labels in tqdm(dataloader):
        images = images.to(device)
        raw_outputs = model(images)
        preds = torch.sigmoid(raw_outputs).cpu() # we don't really care about pre-sigmoid outputs
        all_preds.append(preds)
        all_labels.append(labels)

# collapse predictions and labels into single tensor
all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)

print(f"Predictions shape: {all_preds.shape}")
print(f"Labels shape: {all_labels.shape}")

# Threshold search
For each individual type we collect the optimal threshold, with optimality determined by the Jaccard index

In [ ]:
best_threshold = {}
best_jaccard = {}

for ctype_idx in tqdm(range(len(all_types))):
    best_t = 0
    best_j = 0
    ctype = all_types[ctype_idx]

    preds = all_preds[:, ctype_idx]
    labels = all_labels[:, ctype_idx] == 1
    
    for i in range(1, 100):
        threshold = i * 0.01
        thresholded = preds >= threshold
        
        intersection = (thresholded & labels).sum().item()
        union = (thresholded | labels).sum().item()

        jaccard = intersection / union
        if jaccard > best_j:
            best_t = threshold
            best_j = jaccard

    best_threshold[ctype] = best_t
    best_jaccard[ctype] = best_j

print(best_threshold)
print(best_jaccard)


# Data analysis

In [ ]:
type_counts = Counter(
    t for types_str in df["types"] if isinstance(types_str, str)
    for t in types_str.split("|")
)

total = sum(type_counts[t] for t in all_types)
weighted_jaccard = sum(
    best_jaccard[t] * type_counts[t] / total
    for t in all_types
)

print(f"Weighted Jaccard score: {weighted_jaccard:.3f}")

In [ ]:
# threshold chart
print("thresholds")
fig, ax = plt.subplots(figsize=(14, 5))

sorted_types = sorted(all_types, key=lambda t: best_threshold[t])
sorted_thresholds = [best_threshold[t] for t in sorted_types]

ax.bar(sorted_types, sorted_thresholds)
ax.set_xticks(range(len(sorted_types)))
ax.set_xticklabels(sorted_types, rotation=90)
plt.tight_layout()
plt.show()

# jaccard chart
print("jaccard index")
fig, ax = plt.subplots(figsize=(14, 5))

sorted_types = sorted(all_types, key=lambda t: best_jaccard[t])
sorted_jaccards = [best_jaccard[t] for t in sorted_types]

ax.bar(sorted_types, sorted_jaccards)
ax.set_xticks(range(len(sorted_types)))
ax.set_xticklabels(sorted_types, rotation=90)
plt.tight_layout()
plt.show()

# Export model
This was previously in a separate script. There are two reasons why it has been moved here:
- I decided there is no reason to export a model that has not been evaluated.
- A design change was made: we now have individual thresholds per class. As such we need to export these thresholds with the model. As this is part of the evaluation procedure, it makes sense to export both the model and the thresholds here.

In [ ]:
output_dir = Path(OUTPUT_PATH)
output_dir.mkdir(parents=True, exist_ok=True)

# make sure we pass all supported creature types
with open(output_dir / "classes.json", "w") as f:
    json.dump(all_types, f, indent=2)

# model itself
class FullModel(nn.Module):
    def __init__(self, base_model, thresholds):
        super().__init__()
        self.base = base_model
        self.sigmoid = nn.Sigmoid()

        # this is for baking in the thresholds by shifting the output distribution
        import math
        exponents = [
            math.log(0.5) / math.log(thresholds[i]) for i in range(len(thresholds))
        ]
        # register as a buffer so it's part of the model but not a trainable parameter
        self.register_buffer("exponents", torch.tensor(exponents, dtype=torch.float32))

    def forward(self, x):
        return torch.pow(self.sigmoid(self.base(x)), self.exponents)

ordered_thresholds = [best_threshold[t] for t in all_types]
export_model = FullModel(model, ordered_thresholds)
export_model.eval()
export_model = export_model.to(device)

dummy_input = torch.randn(1, 3, 224, 224).to(device)

torch.onnx.export(
    export_model,
    dummy_input,
    output_dir / "model.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=17
)

# Cleanup
Just to free up memory while the kernel is still open

In [ ]:
del model
gc.collect()
torch.cuda.empty_cache()